# SEMANTIC SIMILARITY

In this project, I will apply what I have learned from my first assignment in the course IN1160 - Introduction to machine learning. 

I will use the "Microsoft Research Paraphrase Corpus" (dataset) from Kaggle.
The dataset includes many different sentences from different news sources from the web. 

# Introduction

I will be make simple vectorrepresentations of words, and these will be used to find similiarities between the words with the use of cosinesimilarity. 

In [113]:
dataset = []

# Part 1: Load the dataset

The format of the dataset is: Sentence ID	String	Author	URL	Agency	Date	Web Date. I only wish to include the "String" (sentence) in my dataset.

In [114]:
def load_dataset(path):
    with open(path, "r") as file:
        header = file.readline().strip().split("\t") #split by tab
        
        index = header.index("String") # find the index of the String column
        #I will make a list of each line by splitting by the tabs..
        #.. and then use this index to access the string

        for line in file:
            row = line.strip().split("\t") #split by tab
            dataset.append(row[index])

    return dataset

In [115]:
dataset = load_dataset("msr_paraphrase_data.txt")

# Part 2: tokens

Tokens are the words we feed into a model. A type is a unique token (word). Our vocabulary includes all our types.

In [116]:
def tokenize(text):
    return text.split() # return a list containing words

In [117]:
def tokenize_dataset(dataset):
    tokenized = [] #a list of lists 
    #the inner lists are sentences, the elements are the words in the respective sentence
    for element in dataset: #element = sentence
        tokenized.append(tokenize(element))
    return tokenized

We will use this tokenized dataset to count the number of types, as well as to make our entire vocabulary which will be used to make the matrix.

**Counting number of tokens**

In [118]:
def nr_token_type(dataset):
    nr_token = 0 #words
    types = set()

    for sentence in tokenize_dataset(dataset):
        for word in sentence:
            types.add(word)
            nr_token += 1
    
    return f"Tokens: {nr_token}. Types: {len(types)}."


print(nr_token_type(dataset))

Tokens: 207476. Types: 28080.


Number of tokens are 207476, but there are 28080 word types. 

# Part 3: Reduce noise / dimensionality

It is important to remove noise when training a model to avoid overfitting. Another way to prevent overfitting is to reduce the dimensionality. I am building a matrix in orde to make the vectors, and I can therefore reduce the number of dimensions by removing unnecessary words; stopwords! 

Stopwords are words that do not contribute much to a text's meaning. I have downloaded the "EN-Stopwords.txt" dataset from Kaggle, and will use this to remove stopwords from the dataset. 

**Stopwords**

In [119]:
def get_stopwords(path):
    stopwords = []
    
    with open(path, "r") as file:
        for line in file:
            stopwords.append(line.strip())
    return stopwords


**Remove stopwords**

In [120]:
stopwords = get_stopwords("EN-Stopwords.txt")

def remove_stopwords(dataset):
    for index, sentence in enumerate(dataset):
        s = sentence.split() #sentence is now a list of words
    
        s = [word for word in s if word.lower() not in stopwords ]
        #wanna make sure that it is case-insensitive
        
        dataset[index] = " ".join(s)
        #the same sentence without the stopwords
    
    return dataset

Let's compare the number of types before and after removing stopwords. 

In [121]:
print("Before removing stopwords:", nr_token_type(dataset))
dataset = remove_stopwords(dataset)
print("After removing stopwords:", nr_token_type(dataset))

Before removing stopwords: Tokens: 207476. Types: 28080.
After removing stopwords: Tokens: 109935. Types: 26995.


Let us now eliminate low-frequency terms, the ones that do not repeat often enough (noise). I will do this by finding the 5000 most common words, and only keep those in the dataset

**Find high frequency words**

In [122]:
from collections import Counter

def common(dataset):
    common = []
    all_words = []

   
    for sentence in tokenize_dataset(dataset):
        all_words.extend(sentence)

   
    five_thousand_most_common = Counter(all_words).most_common(5000)
    #tuple

    for word, nr in five_thousand_most_common:
        common.append(word)

    
    return common


**Remove noise**

In [123]:
def remove_low_freq(common, dataset):
    for index, sentence in enumerate(dataset):
        s = sentence.split() #sentence as a list
        without_common_words = []

        for word in s:
            if word in common:
                without_common_words.append(word)


        dataset[index] = " ".join(without_common_words)

    return dataset

In [124]:
simple_dataset = remove_low_freq(common(dataset), dataset)

In [125]:
print("After removing less frequent words:", nr_token_type(simple_dataset))

After removing less frequent words: Tokens: 72746. Types: 5000.


# Part 4: Vectors

I have now removed noise from the dataset, and use the resulting dataset for my model. I will make vector-representations of the words with a matrix.

## Building our vocabulary

My vocabulary is build up of the different types of words in the dataset.

In [126]:
def build_vocab(dataset):
    vocab = set() 

    for sentence in tokenize_dataset(dataset):
        for word in sentence: 
            vocab.add(word)
    
    return vocab

In [127]:
vocab = build_vocab(dataset) #set

**Type is linked to a specific number**

Because I use a matrix, I have to manage the words with the use of numbers/index.

In [128]:
word_to_int = {}
int_to_word = {} #key = index from word_to_int

for index, word in enumerate(vocab):
    word_to_int[word] = index
    int_to_word[index] = word


I will base the relationship of words on the context they are used in. I will 'assume' that words used in similar contexts often have the same definition / related definitions. 

I will make a matrix where number of rows = number of columns = our vocabulary. So row 1 = first word in our vocabulary. The columns work the same way. The constructor and _create_matrix() in the class "Vectorize" was provided in our assignment in IN1160. 


fit() will update our matrix (initially filled with zeros). It will count the number of instances* two words are used in the same sentence. For example, if the word "cheese" has the row-number 2, and I'm counting how many times it is used in the same sentence as "bread" (which has the column number 3), I will update the position (2,3) in the matrix with the number of instances*.

In [129]:
from numpy import zeros


class Vectorize:
    def __init__(self, vocab, word_to_int):
        self.vocab = vocab
        self.word_to_int = word_to_int
        self.matrix = self._create_matrix(vocab)


    def _create_matrix(self, vocab):

        matrix = zeros((len(vocab), len(vocab)))
        return matrix


    def _add_sentence(self, sentence_tokens):

        #i am assuming that "i" (row) refers to the number that is associated with a word in self.word_to_int
        #one sentence = a list

        for i in range(len(sentence_tokens)):
            for j in range(len(sentence_tokens)):
                if j==i: #skipping the same word
                    continue
                else:
                    self.matrix[word_to_int[sentence_tokens[i]], word_to_int[sentence_tokens[j]]] += 1

    def fit(self, dataset):
        #training the model 
        for sent in dataset:  #sending one list at a time
            self._add_sentence(sent)

    def get_vector(self, word):
        word_int = self.word_to_int[word]
        return self.matrix[word_int]

In [130]:
#vocab = set of all of our wordtypes

vectorizer = Vectorize(vocab, word_to_int)

#training on our data
vectorizer.fit(tokenize_dataset(dataset))

In [131]:
#retrieving a vector for a word and finding out which other word it occurs most often with

random_word = list(vocab)[11] #random 
vector_for_a_word = vectorizer.get_vector(random_word)

max = 0
index = 0

#I iterate over the list to find the word it occurs with the most often
for i in range(len(vector_for_a_word)):
    if vector_for_a_word[i] > max:
        max = vector_for_a_word[i]
        index = i

occurs_most_oftest = int_to_word[index] #finding the word
print(f"{random_word} occurs most often with {occurs_most_oftest}")


Las occurs most often with Vegas,


I have created high-dimensional vectors; they have many dimentions where each vector has space for each word in the vocab. However, it does not necessarily mean that each space have values higher than 0. The vectors are quite detailed, which is a plus (we get lots of information), but it can be heavy to process. 

High-dimensionality can also lead to "curse of dimensionality" which can lead to overfitting! Because they are so large, they might not catch all the necessary information between the words.

# Part 5: Cosine-similarity

I will use cosine-similarity to find how similar two vectors are, and see if they match the similarity seen between words. 



In [132]:
import math

def cosine_similarity(a,b): #a,b are vectors

    a_b = 0 #prikkprodukt (norwegian)

    for A,B in zip(a,b): #creating a tuppel and matching the first element in A to first element in B
        a_b += A*B


    #norm
    squared_A = 0
    for A in a:
        squared_A += A**2

    squared_B = 0

    for B in b:
        squared_B += B**2

    length_a = math.sqrt(squared_A)
    length_b = math.sqrt(squared_B)

    cos = a_b / ((length_a)*(length_b))

    return cos


In [133]:
#example 

#random words:
aa = int_to_word[0] 
bb = int_to_word[10]

a = vectorizer.get_vector(aa)
b = vectorizer.get_vector(bb)

print(f"The words are: {aa} and {bb}. Cosine-similarity: {cosine_similarity(a,b)}")
#values closer to 1 indicates that the words are related to each other. 


The words are: 1.6 and stockholders. Cosine-similarity: 0.033056973204094495


Values closer to 1 means that the vectors point in the same direction. This means that the words (being compared) are similar. The opposite is true for values closer to -1. 

# Part 6: apply the code

I will now apply the code to a list of random 5 words and find the 3 words that are most closely related to each word from another list of random words.

In [134]:
import random

words = random.sample(list(vocab), 5)

#i will use a dictionary for this task

#key = the word from 'words', value = a list of the 3 words that are most closely related
three_closest = []
for w in words:
    three = []

    a = vectorizer.get_vector(w)
    minimum = 2 #cos-value will never be >1
    i = 0 #index


    for word in vocab:
        if w == word: #skip if the same word
            continue
        b = vectorizer.get_vector(word)

        if len(three) < 3: #will add 3 random words
            three.append(word)
            if cosine_similarity(a,b) < minimum: #holding onto minimum value so i can change it later
                minimum = cosine_similarity(a,b)
                i = len(three)-1

        #if list is full and we need to swap out the smallest value
        else:
            if cosine_similarity(a,b) > minimum:
                three[i] = word
                minimum = 2 #update

                for j in range(len(three)):
                    if cosine_similarity(a,vectorizer.get_vector(three[j])) < minimum:
                        minimum = cosine_similarity(a, vectorizer.get_vector(three[j]))
                        i = j


    dictionary = {}
    dictionary[w] = three
    three_closest.append(dictionary)

print(three_closest)


[{'Science': ['three-day', 'Technology', 'Ann']}, {'operation': ['begged', 'laundering', 'cocaine']}, {'weapons': ['war.', "Iraq's", 'biological']}, {'grow': ['2.1', '15.7', '2.9']}, {'life.': ['prostate', 'aggressive', 'develop']}]
